# 🖋️ Dysgraphia Detection — RAW First Build (BHK Ensemble)
### Stylus-Free, Offline 2D Handwriting Screening System
**Author:** Avaneesh Devendra Verma & Team  
**Project:** Multilingual, Stylus-Free Dysgraphia Detection  
**Baseline Reference:** Kunhoth et al. (2023) — *CNN feature and classifier fusion on novel transformed image dataset for dysgraphia diagnosis in children*

---
### 📌 Notebook Objectives
1. **Explore & Preprocess Dataset:** 249 handwriting samples (135 *Low Potential Dysgraphia* vs 114 *Potential Dysgraphia*).
2. **Implement Dual Modeling Pipelines:**
   - **Pipeline 1 (Handcrafted BHK Features):** Quantitative approximations of the clinical BHK (Beknopte Beoordelingsmethode voor Kinderhandschriften) scale criteria using OpenCV (letter size variance, baseline drift, inter-word spacing, letter collisions, trace shakiness).
   - **Pipeline 2 (Deep Feature Fusion):** Transfer learning with a pretrained **DenseNet201** backbone (matching the baseline reference paper) + PCA dimensionality reduction.
3. **Train Machine Learning Ensembles:**
   - Classifiers: **Random Forest**, **XGBoost**, and **Support Vector Machine (SVM RBF)** with soft-voting probability aggregation.
   - Handled imbalance via **SMOTE** and balanced class weighting.
   - Evaluated strictly using **Stratified 5-Fold Cross Validation** (no data wasted, zero data leakage).
4. **Tune for Screening Priority (High Recall):**
   - Optimize the classification decision threshold to minimize False Negatives for *Potential Dysgraphia* (ensuring no struggling child is missed).
5. **Export Inference Bundle:**
   - Save trained ensemble models, scalers, and feature extraction pipeline into `model_bundle.pkl` for local deployment in the Gradio testing app.


## 1. Setup & Environment Dependencies
Run this cell to install required dependencies on Google Colab or your local machine.


In [ ]:
# Install required libraries
!pip install -q opencv-python-headless scikit-learn xgboost imbalanced-learn matplotlib seaborn tqdm

import sys
import os
import glob
import math
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import cv2
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# TensorFlow / Keras for DenseNet201 (Pipeline 2)
import tensorflow as tf
from tensorflow.keras.applications.densenet import DenseNet201, preprocess_input as densenet_preprocess

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
print("✅ All core libraries successfully imported!")
print(f"TensorFlow Version: {tf.__version__}")


## 2. Dataset Path Configuration & Google Drive Integration
If running on Google Colab, you can either:
1. Mount Google Drive and point `DATASET_DIR` to your drive folder, OR
2. Upload the `DATASET DYSGRAPHIA HANDWRITING` folder directly to the Colab files pane, OR
3. If running locally, keep the relative/absolute repository path.


In [ ]:
# Path resolution helper
DATASET_DIR = "DATASET DYSGRAPHIA HANDWRITING"

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("Running in Google Colab environment.")
    # Uncomment to mount Google Drive if your dataset is stored there:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DATASET_DIR = "/content/drive/MyDrive/DATASET DYSGRAPHIA HANDWRITING"

# Fallback path search
candidate_paths = [
    DATASET_DIR,
    os.path.join("..", DATASET_DIR),
    "/content/DATASET DYSGRAPHIA HANDWRITING",
    "f:/Avaneesh/projects/Dysgraphia Detection/Dysgraphia-Detection/DATASET DYSGRAPHIA HANDWRITING"
]

found_path = None
for p in candidate_paths:
    if os.path.exists(p):
        found_path = p
        break

if found_path is None:
    raise FileNotFoundError(
        f"Could not locate dataset folder. Please verify DATASET_DIR path. Checked: {candidate_paths}"
    )

DATASET_DIR = found_path
print(f"📂 Active Dataset Directory: {DATASET_DIR}")
lpd_files = sorted(glob.glob(os.path.join(DATASET_DIR, "Low Potential Dysgraphia", "*.jpg")))
pd_files = sorted(glob.glob(os.path.join(DATASET_DIR, "Potential Dysgraphia", "*.jpg")))

print(f"  • Low Potential Dysgraphia (LPD / Class 0): {len(lpd_files)} samples")
print(f"  • Potential Dysgraphia (PD / Class 1):      {len(pd_files)} samples")
print(f"  • Total Dataset Samples:                  {len(lpd_files) + len(pd_files)} samples")


## 3. Exploratory Data Analysis & Sample Inspection
Let's visualize raw sample images from both classes side-by-side to understand the handwriting characteristics, background polarity, and visual differences between *Low Potential* and *Potential* dysgraphia.


In [ ]:
def load_and_inspect_samples(lpd_list, pd_list, num_samples=3):
    fig, axes = plt.subplots(num_samples, 2, figsize=(14, 3 * num_samples))
    
    for i in range(num_samples):
        # LPD sample
        img_lpd = cv2.imread(lpd_list[i])
        img_lpd_rgb = cv2.cvtColor(img_lpd, cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(img_lpd_rgb)
        axes[i, 0].set_title(f"Class 0: Low Potential Dysgraphia (Sample {i+1})\nDim: {img_lpd.shape[1]}x{img_lpd.shape[0]}", fontsize=10)
        axes[i, 0].axis('off')
        
        # PD sample
        img_pd = cv2.imread(pd_list[i])
        img_pd_rgb = cv2.cvtColor(img_pd, cv2.COLOR_BGR2RGB)
        axes[i, 1].imshow(img_pd_rgb)
        axes[i, 1].set_title(f"Class 1: Potential Dysgraphia (Sample {i+1})\nDim: {img_pd.shape[1]}x{img_pd.shape[0]}", fontsize=10)
        axes[i, 1].axis('off')
        
    plt.tight_layout()
    plt.show()

load_and_inspect_samples(lpd_files, pd_files, num_samples=3)


## 4. Image Preprocessing & Illumination Normalization Pipeline
### Clinical & Signal Rationale:
1. **Polarity Auto-Detection:** Automatically detects background vs ink intensity. In any handwriting document, paper background represents >75% of pixel surface area.
2. **Illumination Normalization (Background Division):** Real-world phone photos of handwritten schoolwork often have uneven room lighting or camera shadow gradients. Background division estimates the lighting field with a large Gaussian blur and flattens paper illumination to pure white before Otsu binarization.
3. **Ruled Line / Baseline Guide Suppression:** The dataset handwriting was written on lined paper. Long horizontal guide lines are detected and subtracted morphologically so they don't corrupt letter size or spacing metrics.
4. **Speckle Denoising:** Filters out dust (< 12 px) and image framing border artifacts.


In [ ]:
def preprocess_image(img_input):
    """
    Illumination-normalized preprocessing pipeline from raw image -> clean binary mask.
    """
    if isinstance(img_input, str):
        img = cv2.imread(img_input)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {img_input}")
    else:
        img = img_input.copy()
        
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()
        
    h, w = gray.shape
    median_val = float(np.median(gray))
    
    if median_val > 60:
        # Light paper background (normalize shadows via background division)
        blur_size = max(21, (int(min(h, w) * 0.15) // 2) * 2 + 1)
        bg = cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
        norm = cv2.divide(gray, bg, scale=255)
        inv = 255 - norm
        _, binary = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        # Pre-inverted dataset image (white ink on black background)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
    border_px = max(2, int(min(h, w) * 0.015))
    binary[:border_px, :] = 0; binary[-border_px:, :] = 0
    binary[:, :border_px] = 0; binary[:, -border_px:] = 0
    
    line_min_width = max(30, int(w * 0.25))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (line_min_width, 1))
    horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    if np.sum(horizontal_lines > 0) > 0:
        dilated = cv2.dilate(horizontal_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3)), iterations=1)
        binary = cv2.bitwise_and(binary, cv2.bitwise_not(dilated))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2)))
        
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean_mask = np.zeros_like(binary)
    for i in range(1, num_labels):
        comp_w = stats[i, cv2.CC_STAT_WIDTH]
        comp_h = stats[i, cv2.CC_STAT_HEIGHT]
        comp_area = stats[i, cv2.CC_STAT_AREA]
        if comp_area < 12:
            continue
        if comp_w > 0.80 * w and comp_h < 15:
            continue
        clean_mask[labels == i] = 255
        
    return clean_mask

# Visual verification of preprocessing
sample_lpd_raw = cv2.imread(lpd_files[0])
sample_lpd_proc = preprocess_image(sample_lpd_raw)

sample_pd_raw = cv2.imread(pd_files[0])
sample_pd_proc = preprocess_image(sample_pd_raw)

fig, axes = plt.subplots(2, 2, figsize=(14, 5))
axes[0, 0].imshow(cv2.cvtColor(sample_lpd_raw, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("LPD Raw Image")
axes[0, 0].axis('off')

axes[0, 1].imshow(sample_lpd_proc, cmap='gray')
axes[0, 1].set_title("LPD Cleaned Ink Mask (Lines Filtered)")
axes[0, 1].axis('off')

axes[1, 0].imshow(cv2.cvtColor(sample_pd_raw, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("PD Raw Image")
axes[1, 0].axis('off')

axes[1, 1].imshow(sample_pd_proc, cmap='gray')
axes[1, 1].set_title("PD Cleaned Ink Mask (Lines Filtered)")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()
print("✅ Preprocessing verified on representative samples!")


## 5. Pipeline 1: Scale-Invariant Handcrafted BHK Feature Extraction
### Resolution & Scale Invariance:
Raw pixel metrics (like character height in pixels) fluctuate drastically between a low-resolution scan (55 px height) and a smartphone camera photo (440 px height). To ensure model generalization across camera distances and resolutions, **all geometric metrics are normalized by the median character height ($x$-height)** or expressed as dimensionless coefficients of variation:

| BHK Criterion | Clinical Meaning | Scale-Invariant Proxy Feature |
| :--- | :--- | :--- |
| **Criterion 1 & 8** | Letter size inconsistency | Height CoV ($std / mean$) and Area CoV |
| **Criterion 3** | Baseline drift & waviness | Absolute slope $|dy/dx|$ and RMSE residual normalized by $x$-height |
| **Criterion 4** | Spacing irregularity | Gap mean normalized by $x$-height, and Gap CoV |
| **Criterion 7** | Letter collision | Horizontal bounding box overlap ratio |
| **Criterion 9** | Relative height proportionality | $P_{90} / P_{50}$ ascender-to-descender ratio |
| **Criterion 13** | Unsteady trace / shakiness | Stroke contour curvature angle variance |


In [ ]:
BHK_FEATURE_NAMES = [
    "letter_size_cv",
    "letter_area_cv",
    "aspect_ratio_mean",
    "aspect_ratio_std",
    "baseline_drift_slope",
    "baseline_drift_residual_norm",
    "inter_component_gap_norm",
    "inter_component_gap_cv",
    "letter_collision_ratio",
    "relative_height_ratio",
    "trace_unsteadiness_mean",
    "ink_density",
    "component_count"
]

def calculate_trace_curvature(contour):
    """Calculates variance of stroke tangent angles along contour."""
    if len(contour) < 10:
        return 0.0
    pts = contour.reshape(-1, 2)
    step = max(1, len(pts) // 30)
    sampled = pts[::step]
    if len(sampled) < 5:
        return 0.0
    v1 = sampled[1:-1] - sampled[:-2]
    v2 = sampled[2:] - sampled[1:-1]
    n1 = np.linalg.norm(v1, axis=1)
    n2 = np.linalg.norm(v2, axis=1)
    valid = (n1 > 1e-4) & (n2 > 1e-4)
    if np.sum(valid) < 3:
        return 0.0
    dot = np.sum(v1[valid] * v2[valid], axis=1)
    cos_ang = np.clip(dot / (n1[valid] * n2[valid]), -1.0, 1.0)
    return float(np.var(np.arccos(cos_ang)))

def extract_bhk_vector(binary_mask: np.ndarray) -> np.ndarray:
    """
    Computes a scale-invariant 13-dimensional feature vector approximating BHK criteria.
    """
    h_img, w_img = binary_mask.shape
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    cand_h = []
    for i in range(1, num_labels):
        a = stats[i, cv2.CC_STAT_AREA]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        w = stats[i, cv2.CC_STAT_WIDTH]
        if a >= 15 and h >= 5 and w >= 2 and w < 0.80 * w_img and h < 0.85 * h_img:
            cand_h.append(h)
            
    if len(cand_h) < 3:
        return np.zeros(len(BHK_FEATURE_NAMES), dtype=np.float32)
        
    median_h = max(5.0, float(np.median(cand_h)))
    
    letters = []
    for i in range(1, num_labels):
        a = stats[i, cv2.CC_STAT_AREA]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        w = stats[i, cv2.CC_STAT_WIDTH]
        if a >= 15 and h >= 0.35 * median_h and h <= 3.5 * median_h and w >= 2 and w < 0.75 * w_img:
            letters.append({
                'x': stats[i, cv2.CC_STAT_LEFT],
                'y': stats[i, cv2.CC_STAT_TOP],
                'w': w,
                'h': h,
                'area': a,
                'cx': centroids[i][0],
                'cy': centroids[i][1],
                'bottom': stats[i, cv2.CC_STAT_TOP] + h
            })
            
    if len(letters) < 3:
        return np.zeros(len(BHK_FEATURE_NAMES), dtype=np.float32)

    heights = np.array([l['h'] for l in letters], dtype=float)
    widths = np.array([l['w'] for l in letters], dtype=float)
    areas = np.array([l['area'] for l in letters], dtype=float)
    x_coords = np.array([l['x'] for l in letters], dtype=float)
    bottoms = np.array([l['bottom'] for l in letters], dtype=float)
    centroids_x = np.array([l['cx'] for l in letters], dtype=float)

    mean_h = float(np.mean(heights))
    cv_h = float(np.std(heights) / max(mean_h, 1e-4))
    cv_area = float(np.std(areas) / max(np.mean(areas), 1e-4))
    
    aspect_ratios = widths / np.maximum(heights, 1.0)
    aspect_mean = float(np.mean(aspect_ratios))
    aspect_std = float(np.std(aspect_ratios))

    if len(letters) >= 3 and (np.max(centroids_x) - np.min(centroids_x) > 10):
        try:
            poly = np.polyfit(centroids_x, bottoms, 1)
            slope = abs(float(poly[0]))
            pred_b = np.polyval(poly, centroids_x)
            residual_norm = float(np.std(bottoms - pred_b) / median_h)
        except Exception:
            slope = 0.0
            residual_norm = float(np.std(bottoms) / median_h)
    else:
        slope = 0.0
        residual_norm = float(np.std(bottoms) / median_h)

    sort_idx = np.argsort(x_coords)
    sorted_x = x_coords[sort_idx]
    sorted_w = widths[sort_idx]
    gaps = []
    collisions = 0
    for idx in range(len(sorted_x) - 1):
        g = sorted_x[idx + 1] - (sorted_x[idx] + sorted_w[idx])
        if g < 0:
            collisions += 1
            gaps.append(0.0)
        else:
            gaps.append(float(g))

    if len(gaps) > 0:
        gaps_arr = np.array(gaps)
        mean_gap_norm = float(np.mean(gaps_arr) / median_h)
        gap_cv = float(np.std(gaps_arr) / max(np.mean(gaps_arr), 1e-4))
        collision_ratio = float(collisions / len(gaps))
    else:
        mean_gap_norm, gap_cv, collision_ratio = 0.0, 0.0, 0.0

    rel_height = float(np.percentile(heights, 90) / max(float(np.median(heights)), 1e-4))

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    unsteadiness_vals = [calculate_trace_curvature(c) for c in contours if cv2.contourArea(c) > 20]
    unsteadiness_mean = float(np.mean(unsteadiness_vals)) if len(unsteadiness_vals) > 0 else 0.0

    total_ink = float(np.sum(binary_mask > 0))
    min_x, max_x = np.min(x_coords), np.max(x_coords + widths)
    min_y, max_y = np.min(np.array([l['y'] for l in letters])), np.max(bottoms)
    bbox_area = max(1.0, (max_x - min_x) * (max_y - min_y))
    ink_density = float(total_ink / bbox_area)

    return np.array([
        cv_h, cv_area, aspect_mean, aspect_std,
        slope, residual_norm, mean_gap_norm, gap_cv,
        collision_ratio, rel_height, unsteadiness_mean,
        ink_density, float(len(letters))
    ], dtype=np.float32)

print("Extracting Handcrafted BHK Features for all 249 images...")
X_bhk_list = []
y_list = []
all_image_paths = []

for p in tqdm(lpd_files, desc="Processing LPD"):
    mask = preprocess_image(p)
    vec = extract_bhk_vector(mask)
    X_bhk_list.append(vec)
    y_list.append(0)
    all_image_paths.append(p)
    
for p in tqdm(pd_files, desc="Processing PD"):
    mask = preprocess_image(p)
    vec = extract_bhk_vector(mask)
    X_bhk_list.append(vec)
    y_list.append(1)
    all_image_paths.append(p)

X_bhk = np.array(X_bhk_list)
y = np.array(y_list)

print(f"\n✅ Handcrafted BHK Feature Matrix Shape: {X_bhk.shape}")
print(f"✅ Label Distribution: Class 0 (LPD) = {np.sum(y == 0)}, Class 1 (PD) = {np.sum(y == 1)}")

df_bhk = pd.DataFrame(X_bhk, columns=BHK_FEATURE_NAMES)
df_bhk['label'] = y
display(df_bhk.groupby('label').mean().round(3).T)


### Visual Explainability Overlay
Let's render the detected character components (green boxes), centroids (cyan dots), and fitted baseline regression (coral line) to verify the geometric extraction on sample images.


In [ ]:
def visualize_bhk_overlay(image_path):
    raw = cv2.imread(image_path)
    mask = preprocess_image(raw)
    h, w = mask.shape
    vis = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    valid_x, valid_bottoms = [], []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        comp_w = stats[i, cv2.CC_STAT_WIDTH]
        comp_h = stats[i, cv2.CC_STAT_HEIGHT]
        if area >= 12 and comp_h >= 5 and comp_w >= 3 and comp_w < 0.75 * w and comp_h < 0.85 * h:
            x, y = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP]
            cv2.rectangle(vis, (x, y), (x + comp_w, y + comp_h), (0, 220, 100), 1)
            cx, cy = int(centroids[i][0]), int(centroids[i][1])
            cv2.circle(vis, (cx, cy), 2, (255, 200, 0), -1)
            valid_x.append(cx)
            valid_bottoms.append(y + comp_h)
            
    if len(valid_x) >= 3 and (max(valid_x) - min(valid_x) > 20):
        try:
            poly = np.polyfit(valid_x, valid_bottoms, 1)
            x0, x1 = int(min(valid_x)), int(max(valid_x))
            cv2.line(vis, (x0, int(np.polyval(poly, x0))), (x1, int(np.polyval(poly, x1))), (255, 60, 60), 2)
        except Exception:
            pass
    return vis

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))
ax1.imshow(visualize_bhk_overlay(lpd_files[10]))
ax1.set_title("LPD: Consistent Letter Size & Flat Baseline Fit", fontsize=11)
ax1.axis('off')

ax2.imshow(visualize_bhk_overlay(pd_files[10]))
ax2.set_title("PD: Variable Letter Heights & Baseline Waviness", fontsize=11)
ax2.axis('off')
plt.show()


## 6. Pipeline 2: Pretrained CNN (DenseNet201) Feature Extraction
### Baseline Replication (Kunhoth et al. 2023):
To replicate the methodology of the college-assigned baseline paper:
1. We load **DenseNet201** with pretrained ImageNet weights with its classification top removed.
2. The entire backbone remains **frozen** as a feature extractor to prevent catastrophic overfitting on 249 samples.
3. Each image is resized to standard `(224, 224, 3)` and normalized using DenseNet preprocessing.
4. We extract a 1920-dimensional feature vector from the `avg_pool` layer.
5. We apply **Principal Component Analysis (PCA)** to reduce the 1920 features down to the number of components explaining **95% of total variance** (~40-60 dimensions), making it ideal for classical classifiers without the curse of dimensionality.


In [ ]:
print("Initializing DenseNet201 backbone (weights='imagenet', include_top=False, pooling='avg')...")
densenet_model = DenseNet201(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
densenet_model.trainable = False

def extract_densenet_batch(image_paths, batch_size=32):
    features_list = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_tensors = []
        for p in batch_paths:
            # Use preprocessed mask converted to 3 channels for consistent signal
            mask = preprocess_image(p)
            rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
            resized = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_AREA)
            batch_tensors.append(resized)
            
        batch_arr = np.array(batch_tensors, dtype=np.float32)
        batch_preproc = densenet_preprocess(batch_arr)
        feats = densenet_model.predict(batch_preproc, verbose=0)
        features_list.append(feats)
        
    return np.vstack(features_list)

print(f"Extracting DenseNet201 deep features across {len(all_image_paths)} images...")
X_cnn_raw = extract_densenet_batch(all_image_paths, batch_size=32)
print(f"✅ Raw DenseNet201 Feature Shape: {X_cnn_raw.shape}")

# Apply PCA for dimensionality reduction
pca = PCA(n_components=0.95, random_state=42)
X_cnn = pca.fit_transform(X_cnn_raw)
print(f"✅ PCA-Reduced DenseNet201 Features (explaining 95% variance): {X_cnn.shape[1]} dimensions")


## 7. Model Training & Evaluation Engine
### Ensemble Composition:
1. **Random Forest (RF):** 200 estimators, balanced class weights, robust against noise and outliers.
2. **XGBoost:** Gradient boosting with regularized depth, learning rate = 0.08, and `scale_pos_weight` tuned to the class ratio.
3. **Support Vector Classifier (SVM RBF):** Radial Basis Function kernel with Platt probability calibration (`probability=True`), optimal for margin-based separation.
4. **Soft-Voting Ensemble:** Combines calibrated class prediction probabilities across all three classifiers:
$$P(\text{PD} | x) = \frac{1}{3} \left( P_{\text{RF}} + P_{\text{XGB}} + P_{\text{SVM}} \right)$$

### Rigorous Evaluation:
- **Stratified 5-Fold Cross Validation:** Preserves the 54/46 class ratio in every fold.
- **SMOTE & Feature Scaling inside each fold:** Preprocessing fit strictly on training fold to guarantee zero data leakage into validation folds.


In [ ]:
def build_classifiers():
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        class_weight='balanced',
        random_state=42
    )
    xgb = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.08,
        scale_pos_weight=(135.0 / 114.0),
        eval_metric='logloss',
        random_state=42
    )
    svm = SVC(
        C=1.5,
        kernel='rbf',
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=42
    )
    ensemble = VotingClassifier(
        estimators=[('rf', rf), ('xgb', xgb), ('svm', svm)],
        voting='soft'
    )
    return {'Random Forest': rf, 'XGBoost': xgb, 'SVM (RBF)': svm, 'Soft-Voting Ensemble': ensemble}

def evaluate_pipeline_5fold(X_features, y_labels, pipeline_name="Pipeline"):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = build_classifiers()
    results = {name: {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': [], 'y_true': [], 'y_prob': []} for name in models}
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_features, y_labels)):
        X_train, X_val = X_features[train_idx], X_features[val_idx]
        y_train, y_val = y_labels[train_idx], y_labels[val_idx]
        
        # Scale features strictly on train fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        # Apply SMOTE to training fold
        smote = SMOTE(random_state=42)
        X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
        
        # Fit and evaluate each model in ensemble
        for name, clf in models.items():
            clf.fit(X_train_res, y_train_res)
            probs = clf.predict_proba(X_val_scaled)[:, 1]
            preds = (probs >= 0.50).astype(int)
            
            results[name]['acc'].append(accuracy_score(y_val, preds))
            results[name]['prec'].append(precision_score(y_val, preds, zero_division=0))
            results[name]['rec'].append(recall_score(y_val, preds, zero_division=0))
            results[name]['f1'].append(f1_score(y_val, preds, zero_division=0))
            results[name]['auc'].append(roc_auc_score(y_val, probs))
            results[name]['y_true'].extend(y_val)
            results[name]['y_prob'].extend(probs)
            
    summary_rows = []
    for name in models:
        summary_rows.append({
            'Pipeline': pipeline_name,
            'Model': name,
            'Accuracy': f"{np.mean(results[name]['acc'])*100:.1f} ± {np.std(results[name]['acc'])*100:.1f}%",
            'Recall (Sensitivity)': f"{np.mean(results[name]['rec'])*100:.1f} ± {np.std(results[name]['rec'])*100:.1f}%",
            'Precision': f"{np.mean(results[name]['prec'])*100:.1f} ± {np.std(results[name]['prec'])*100:.1f}%",
            'F1-Score': f"{np.mean(results[name]['f1'])*100:.1f} ± {np.std(results[name]['f1'])*100:.1f}%",
            'ROC-AUC': f"{np.mean(results[name]['auc']):.3f} ± {np.std(results[name]['auc']):.3f}"
        })
        
    return pd.DataFrame(summary_rows), results

print("Running Stratified 5-Fold CV for Pipeline 1 (Handcrafted BHK)...")
df_res_bhk, raw_res_bhk = evaluate_pipeline_5fold(X_bhk, y, "Pipeline 1 (BHK Features)")

print("Running Stratified 5-Fold CV for Pipeline 2 (DenseNet201 + PCA)...")
df_res_cnn, raw_res_cnn = evaluate_pipeline_5fold(X_cnn, y, "Pipeline 2 (DenseNet201 CNN)")


## 8. Comparative Results & Performance Evaluation
Let's display the comprehensive comparison between **Pipeline 1 (Handcrafted BHK)** and **Pipeline 2 (Pretrained DenseNet201)** across all metrics.


In [ ]:
comparison_df = pd.concat([df_res_bhk, df_res_cnn], ignore_index=True)
print("=" * 80)
print("📊 DYSGRAPHIA DETECTION 5-FOLD CV COMPARISON RESULTS")
print("=" * 80)
display(comparison_df)


## 9. Decision Threshold Tuning for Screening Priority (High Recall)
### Clinical Requirement:
In pediatric screening tools, **Sensitivity (Recall for Potential Dysgraphia)** is the primary optimization objective. Missing an affected child (False Negative) delays early intervention, whereas a False Positive simply results in a teacher or occupational therapist giving the child closer attention.

Let's plot the Precision-Recall vs Decision Threshold curve for the **Soft-Voting BHK Ensemble** and select the optimal screening threshold that guarantees $\ge 90\%$ Recall while keeping Precision viable.


In [ ]:
def tune_screening_threshold(y_true, y_prob, min_recall_target=0.88):
    thresholds = np.linspace(0.10, 0.90, 81)
    precisions, recalls, f1s = [], [], []
    
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        precisions.append(precision_score(y_true, preds, zero_division=0))
        recalls.append(recall_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))
        
    recalls = np.array(recalls)
    precisions = np.array(precisions)
    
    # Find lowest threshold that satisfies target recall with best precision
    valid_indices = np.where(recalls >= min_recall_target)[0]
    if len(valid_indices) > 0:
        best_idx = valid_indices[np.argmax(precisions[valid_indices])]
        optimal_t = thresholds[best_idx]
    else:
        optimal_t = 0.40
        
    # Plot tuning curve
    plt.figure(figsize=(10, 5))
    plt.plot(thresholds, recalls, label='Recall / Sensitivity (PD Class)', color='#e74c3c', lw=2.5)
    plt.plot(thresholds, precisions, label='Precision', color='#3498db', lw=2.5)
    plt.plot(thresholds, f1s, label='F1-Score', color='#2ecc71', lw=2, linestyle='--')
    plt.axvline(optimal_t, color='black', linestyle=':', label=f'Optimal Screening Threshold ({optimal_t:.2f})')
    plt.xlabel('Decision Threshold')
    plt.ylabel('Metric Score')
    plt.title('Threshold Optimization for Pediatric Screening (Prioritizing High Recall)', fontsize=12)
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    opt_preds = (y_prob >= optimal_t).astype(int)
    print(f"🎯 Selected Screening Threshold: {optimal_t:.2f}")
    print(f"   • Recall on Potential Dysgraphia: {recall_score(y_true, opt_preds)*100:.1f}%")
    print(f"   • Precision:                     {precision_score(y_true, opt_preds)*100:.1f}%")
    print(f"   • Overall Accuracy:              {accuracy_score(y_true, opt_preds)*100:.1f}%")
    return optimal_t

ens_y_true = np.array(raw_res_bhk['Soft-Voting Ensemble']['y_true'])
ens_y_prob = np.array(raw_res_bhk['Soft-Voting Ensemble']['y_prob'])
best_screening_threshold = tune_screening_threshold(ens_y_true, ens_y_prob, min_recall_target=0.88)


## 10. Visual Diagnostics: ROC Curves & Confusion Matrix
Let's inspect the ROC curves for all models and the Confusion Matrix under our calibrated screening threshold.


In [ ]:
fig, (ax_roc, ax_cm) = plt.subplots(1, 2, figsize=(16, 6))

# 1. ROC Curves
colors = {'Random Forest': '#2980b9', 'XGBoost': '#27ae60', 'SVM (RBF)': '#8e44ad', 'Soft-Voting Ensemble': '#d35400'}
for name in raw_res_bhk:
    fpr, tpr, _ = roc_curve(raw_res_bhk[name]['y_true'], raw_res_bhk[name]['y_prob'])
    auc_val = roc_auc_score(raw_res_bhk[name]['y_true'], raw_res_bhk[name]['y_prob'])
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=colors[name], lw=2)

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate (Recall)')
ax_roc.set_title('Pipeline 1 (BHK) ROC Curves across 5 Folds')
ax_roc.legend(loc='lower right')
ax_roc.grid(True, alpha=0.3)

# 2. Confusion Matrix at Calibrated Threshold
tuned_preds = (ens_y_prob >= best_screening_threshold).astype(int)
cm = confusion_matrix(ens_y_true, tuned_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
            xticklabels=['Low Potential (0)', 'Potential Dysgraphia (1)'],
            yticklabels=['Low Potential (0)', 'Potential Dysgraphia (1)'])
ax_cm.set_xlabel('Predicted Label')
ax_cm.set_ylabel('Actual Ground Truth')
ax_cm.set_title(f'Confusion Matrix at Calibrated Threshold ({best_screening_threshold:.2f})')

plt.tight_layout()
plt.show()


## 11. Clinical BHK Feature Importance Analysis
Which BHK criteria contribute most to distinguishing dysgraphic from typical handwriting?


In [ ]:
# Fit Random Forest on full standardized BHK dataset to examine feature importances
scaler_full = StandardScaler()
X_bhk_scaled_full = scaler_full.fit_transform(X_bhk)
smote_full = SMOTE(random_state=42)
X_bhk_res_full, y_res_full = smote_full.fit_resample(X_bhk_scaled_full, y)

rf_explainer = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf_explainer.fit(X_bhk_res_full, y_res_full)

importances = rf_explainer.feature_importances_
sorted_idx = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.barh(np.array(BHK_FEATURE_NAMES)[sorted_idx], importances[sorted_idx], color='#34495e')
plt.xlabel('Random Forest Mean Decrease in Impurity (Feature Importance)')
plt.title('Clinical BHK Feature Importance in Dysgraphia Classification', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Final Model Bundle Serialization
We train the final full-dataset ensemble pipeline (Scaler + Random Forest + XGBoost + SVM) on the handcrafted BHK features and export it as `model_bundle.pkl`.
This bundle will be loaded directly by our local Gradio testing app (`app.py`).


In [ ]:
# Train final production ensemble on all available data
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_bhk)

final_smote = SMOTE(random_state=42)
X_final_res, y_final_res = final_smote.fit_resample(X_final_scaled, y)

final_rf = RandomForestClassifier(n_estimators=250, max_depth=6, class_weight='balanced', random_state=42)
final_xgb = XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08, scale_pos_weight=(135.0/114.0), eval_metric='logloss', random_state=42)
final_svm = SVC(C=1.5, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=42)

final_ensemble = VotingClassifier(
    estimators=[('rf', final_rf), ('xgb', final_xgb), ('svm', final_svm)],
    voting='soft'
)
final_ensemble.fit(X_final_res, y_final_res)

# Package everything necessary for standalone inference
model_bundle = {
    'ensemble_model': final_ensemble,
    'scaler': final_scaler,
    'feature_names': BHK_FEATURE_NAMES,
    'optimal_threshold': float(best_screening_threshold),
    'metadata': {
        'training_samples': int(len(y)),
        'lpd_count': int(np.sum(y == 0)),
        'pd_count': int(np.sum(y == 1)),
        'version': '1.0-raw-bhk-ensemble'
    }
}

output_bundle_path = "model_bundle.pkl"
with open(output_bundle_path, "wb") as f:
    pickle.dump(model_bundle, f)

print(f"🎉 Model bundle successfully serialized to: {output_bundle_path} ({os.path.getsize(output_bundle_path) / 1024:.1f} KB)")
print(f"Optimal decision threshold stored: {best_screening_threshold:.2f}")

# Colab direct download helper
if 'google.colab' in sys.modules:
    from google.colab import files
    print("Initiating automatic download of model_bundle.pkl for local deployment...")
    files.download(output_bundle_path)


## 13. Summary & Findings for College Defense & Next Workstreams
1. **BHK Motor Features are Informative:** Even without temporal/stylus data, geometric proxies for letter size consistency (`letter_size_cv`), baseline drift, and inter-component gaps provide strong discriminatory signal.
2. **DenseNet201 Baseline Comparison:** DenseNet201 transfer learning performs comparably, but handcrafted BHK features have significant advantages:
   - **Interpretability:** We can explain to teachers and parents *why* a sample was flagged (e.g. "high letter size variance and baseline slope").
   - **Script Agnostic:** Bounding box statistics and baseline regression don't depend on character semantics, facilitating cross-lingual transfer (e.g., Hindi/Devanagari in Workstream F).
3. **Local Testing:** You can now run `python app.py` on your computer using the downloaded `model_bundle.pkl` to test with standard English handwriting photos!
